____

# Advanced Retrieval-Augmented Generation (RAG) and Language Model Implementation

## Educational Implementation and Analysis

<h2>IMPORTANT DISCLAIMER</h2>

This notebook is developed for personal learning and educational purposes only.
It is not intended for commercial use or distribution. The implementation combines
multiple AI techniques and serves as a comprehensive study of modern RAG systems.

The document corpus used consists of publicly available NASA educational materials,
used here for demonstration and learning purposes in accordance with fair use principles.

- Author: Personal Learning Project
- Purpose: Educational exploration of RAG systems and AI text generation

In [1]:
# =============================================================================
# SECTION 1: DEPENDENCIES AND SYSTEM INITIALIZATION
# =============================================================================

# Installation requirements (run this cell first if packages are not installed)
# pip install torch transformers sentence-transformers faiss-cpu langchain langchain-community pypdf2 rank-bm25 numpy

import torch
import numpy as np
import faiss
import re
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi
import warnings

warnings.filterwarnings("ignore")

print("All dependencies loaded successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

All dependencies loaded successfully
PyTorch version: 2.8.0+cu128
CUDA available: True


In [2]:
# Advanced Retrieval-Augmented Generation (RAG) Systems and AI Text Generation
# Educational Implementation and Analysis

"""
IMPORTANT DISCLAIMER:
This notebook is developed for personal learning and educational purposes only.
It is not intended for commercial use or distribution. The implementation combines
multiple AI techniques and serves as a comprehensive study of modern RAG systems.

The document corpus used consists of publicly available NASA educational materials,
used here for demonstration and learning purposes in accordance with fair use principles.

Author: Personal Learning Project
Purpose: Educational exploration of RAG systems and AI text generation
License: Educational/Non-commercial use only
"""

# %%
# =============================================================================
# SECTION 1: DEPENDENCIES AND SYSTEM INITIALIZATION
# =============================================================================

"""
This section handles the installation and import of all required libraries.
We use a comprehensive set of modern AI/ML libraries for implementing
state-of-the-art RAG systems.
"""

# Installation requirements (run this cell first if packages are not installed)
# pip install torch transformers sentence-transformers faiss-cpu langchain langchain-community pypdf2 rank-bm25 numpy

import torch
import numpy as np
import faiss
import re
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi
import warnings

warnings.filterwarnings("ignore")

print("All dependencies loaded successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# %%
# =============================================================================
# SECTION 2: MODEL CONFIGURATION AND SELECTION
# =============================================================================

"""
Model Selection Strategy for Resource-Constrained Environments:

For educational purposes and systems with limited GPU memory (8GB CUDA), we carefully
select models that balance performance with computational efficiency. The Qwen2.5 
series provides excellent reasoning capabilities while maintaining reasonable memory footprints.

Model Options Analysis:
- Qwen/Qwen2.5-0.5B-Instruct: Ultra-conservative choice (0.5B parameters)
- Qwen/Qwen2.5-1.5B-Instruct: Balanced choice for educational use (1.5B parameters)  
- Qwen/Qwen2.5-3B-Instruct: Maximum recommended for 8GB systems (3B parameters)

We select the 1.5B model as it provides optimal balance between capability and 
resource requirements for educational exploration.
"""

# Model configuration
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"


def initialize_system():
    """
    Initialize the complete AI system with memory optimization.

    This function sets up:
    1. Language model for text generation with memory-efficient settings
    2. Sentence transformer for embedding generation
    3. Proper device allocation and memory management

    Returns:
        tuple: (tokenizer, model, embedder) - The core system components
    """
    print("Initializing AI system components...")

    # Initialize tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

    # Initialize language model with memory optimization
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float16,  # Use float16 for memory efficiency
        device_map="cuda" if torch.cuda.is_available() else "cpu",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )

    # Initialize embedding model for semantic search
    embedder = SentenceTransformer("all-MiniLM-L6-v2")

    return tokenizer, model, embedder


# Initialize system components
tokenizer, model, embedder = initialize_system()
print(f"System initialized successfully with {MODEL_NAME}")

All dependencies loaded successfully
PyTorch version: 2.8.0+cu128
CUDA available: True
Initializing AI system components...
System initialized successfully with Qwen/Qwen2.5-0.5B-Instruct


In [3]:
# =============================================================================
# SECTION 3: TEXT GENERATION ENGINE
# =============================================================================

"""
Text Generation Theory and Implementation:

Text generation in modern language models relies on probabilistic sampling from
the model's learned distribution over possible next tokens. Key parameters include:

1. Temperature (τ): Controls randomness in sampling
   - Formula: softmax(logits/τ)
   - Lower values (0.1-0.3): More deterministic, focused responses
   - Higher values (0.7-1.0): More creative, diverse responses

2. Top-k Sampling: Limits sampling to k most probable tokens
3. Top-p (Nucleus) Sampling: Samples from smallest set with cumulative probability p
4. Repetition Penalty: Reduces likelihood of repeating recent tokens

These techniques work together to produce coherent, diverse, and contextually
appropriate text while avoiding common issues like repetition and incoherence.
"""


def generate_response(prompt, max_tokens=350, temperature=0.1):
    """
    Generate text response using the language model with optimized parameters.

    This function implements a complete text generation pipeline with:
    - Proper prompt formatting for instruction-following models
    - Memory-efficient generation with gradient disabled
    - Error handling for robust operation

    Args:
        prompt (str): Input text prompt
        max_tokens (int): Maximum number of tokens to generate
        temperature (float): Sampling temperature for creativity control

    Returns:
        str: Generated text response
    """
    try:
        # Format prompt for instruction-following model
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        # Tokenize input with proper truncation
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # Generate response with memory efficiency
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temperature,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Decode and return clean response
        response = tokenizer.decode(
            outputs[0][len(inputs["input_ids"][0]) :], skip_special_tokens=True
        )
        return response.strip()

    except Exception as e:
        return f"Generation error: {str(e)}"


# Test the text generation system
test_response = generate_response(
    "Explain the concept of artificial intelligence in simple terms."
)
print("Text Generation Test:")
print(f"Response: {test_response}")

Text Generation Test:
Response: Artificial Intelligence (AI) is a field that focuses on creating machines and software that can perform tasks that typically require human intelligence, such as learning, problem-solving, perception, reasoning, and decision-making. In other words, AI aims to create intelligent systems that can think like humans or even surpass them in certain areas.

The concept of AI has been around for decades, but it gained significant traction in recent years due to advancements in machine learning algorithms and deep neural networks. These technologies allow computers to learn from data and improve their performance over time, making them capable of performing complex tasks with increasing accuracy and efficiency.

Some key aspects of AI include:

1. Machine Learning: This involves training computer systems using large datasets to identify patterns and make predictions.
2. Deep Learning: A subset of machine learning that uses neural networks with multiple layers to 

In [4]:
# =============================================================================
# SECTION 4: DOCUMENT PROCESSING PIPELINE
# =============================================================================

"""
Document Processing Theory:

Effective RAG systems require sophisticated document processing to create
meaningful, searchable chunks of information. Our pipeline implements:

1. Document Loading: Extract text from various formats (PDF, TXT)
2. Text Chunking: Split documents into semantically coherent segments
   - Recursive splitting preserves context while maintaining size limits
   - Overlap between chunks ensures no information loss at boundaries
3. Embedding Generation: Convert text chunks to dense vector representations
4. Index Creation: Build efficient search structures for fast retrieval

The RecursiveCharacterTextSplitter uses a hierarchy of separators:
["\n\n", "\n", ". ", " "] to maintain semantic coherence while respecting
size constraints.
"""


def process_documents(docs_path="./documents"):
    """
    Comprehensive document processing pipeline for RAG systems.

    This function creates a complete knowledge base by:
    1. Loading documents from specified directory
    2. Splitting text into optimally-sized chunks
    3. Generating embeddings for semantic search
    4. Creating search indices for efficient retrieval

    Args:
        docs_path (str): Path to directory containing documents

    Returns:
        dict: Complete document system with texts, embeddings, and indices
    """

    print(f"Processing documents from: {docs_path}")

    # Step 1: Load documents
    documents = []
    docs_directory = Path(docs_path)

    if not docs_directory.exists():
        print(
            f"Warning: Directory {docs_path} not found. Creating example documents..."
        )
        docs_directory.mkdir(exist_ok=True)

        # Create sample NASA-style educational content
        sample_content = """
        NASA's Mission to Mars: Understanding Rocket Propulsion
        
        Rocket propulsion operates on Newton's third law of motion: for every action, 
        there is an equal and opposite reaction. When propellant is expelled from a 
        rocket engine at high velocity, it creates thrust that propels the spacecraft forward.
        
        The basic rocket equation, developed by Konstantin Tsiolkovsky, describes the
        relationship between rocket mass, exhaust velocity, and achievable velocity:
        Δv = ve * ln(m0/mf)
        
        Where:
        - Δv is the change in velocity
        - ve is the effective exhaust velocity
        - m0 is the initial mass
        - mf is the final mass
        
        This fundamental equation governs all rocket design and mission planning for
        interplanetary travel.
        """

        with open(docs_directory / "sample_nasa_content.txt", "w") as f:
            f.write(sample_content)

    # Load PDF documents
    for pdf_file in docs_directory.glob("*.pdf"):
        try:
            loader = PyPDFLoader(str(pdf_file))
            pages = loader.load()
            for page in pages:
                page.metadata.update({"source": pdf_file.name})
                documents.append(page)
            print(f"Loaded: {pdf_file.name}")
        except Exception as e:
            print(f"Error loading {pdf_file}: {e}")

    # Load text documents
    for txt_file in docs_directory.glob("*.txt"):
        try:
            with open(txt_file, "r", encoding="utf-8") as f:
                content = f.read()
                # Create document-like object
                doc_obj = type(
                    "Document",
                    (),
                    {"page_content": content, "metadata": {"source": txt_file.name}},
                )()
                documents.append(doc_obj)
            print(f"Loaded: {txt_file.name}")
        except Exception as e:
            print(f"Error loading {txt_file}: {e}")

    if not documents:
        raise ValueError(
            "No documents found. Please add PDF or TXT files to the documents directory."
        )

    # Step 2: Split documents into chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=750,  # Optimal size for context preservation
        chunk_overlap=150,  # Overlap to prevent information loss
        separators=["\n\n", "\n", ". ", " "],  # Hierarchical splitting
    )
    chunks = splitter.split_documents(documents)
    texts = [chunk.page_content for chunk in chunks]

    # Step 3: Generate embeddings for semantic search
    print("Generating embeddings for semantic search...")
    embeddings = embedder.encode(texts, batch_size=16, show_progress_bar=False)

    # Step 4: Create FAISS index for efficient similarity search
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner product for cosine similarity
    faiss.normalize_L2(embeddings.astype("float32"))
    index.add(embeddings.astype("float32"))

    # Step 5: Create BM25 index for keyword search
    tokenized_texts = [re.findall(r"\w+", text.lower()) for text in texts]
    bm25 = BM25Okapi(tokenized_texts)

    print(f"Document processing complete:")
    print(f"- Total chunks: {len(texts)}")
    print(f"- Embedding dimension: {dim}")
    print(
        f"- Average chunk length: {np.mean([len(text) for text in texts]):.0f} characters"
    )

    return {
        "texts": texts,
        "chunks": chunks,
        "embeddings": embeddings,
        "faiss_index": index,
        "bm25_index": bm25,
        "tokenized_texts": tokenized_texts,
    }


# Process documents and create knowledge base
doc_system = process_documents()

Processing documents from: ./documents
Loaded: main_ShuttleRetrospective.pdf
Loaded: NASA_exploration.pdf
Loaded: rockets-educator-guide-20.pdf
Generating embeddings for semantic search...
Document processing complete:
- Total chunks: 1051
- Embedding dimension: 384
- Average chunk length: 574 characters


In [5]:
# =============================================================================
# SECTION 5: UNIVERSAL SEARCH SYSTEM
# =============================================================================

"""
Multi-Modal Search Theory:

Modern information retrieval systems combine multiple search paradigms to
maximize recall and precision. Our universal search system implements:

1. Semantic Search (Dense Retrieval):
   - Uses neural embeddings to capture semantic similarity
   - Effective for conceptual queries and paraphrasing
   - Formula: similarity = cosine(query_embedding, document_embedding)

2. Keyword Search (Sparse Retrieval):
   - BM25 algorithm for exact term matching
   - Effective for specific terms, names, and precise queries
   - Formula: BM25(q,d) = Σ IDF(qi) * (tf(qi,d) * (k1+1)) / (tf(qi,d) + k1 * (1-b+b*|d|/avgdl))

3. Hybrid Search:
   - Combines both approaches for comprehensive coverage
   - Balances semantic understanding with precise term matching
   - Provides robustness across different query types

The system normalizes scores from different methods and removes duplicates
to provide a unified, ranked result set.
"""


def universal_search(query, method="hybrid", k=5, threshold=0.6):
    """
    Universal search function supporting multiple retrieval methods.

    This function implements three search strategies:
    1. Semantic search using dense embeddings
    2. Keyword search using BM25 algorithm
    3. Hybrid search combining both approaches

    Args:
        query (str): Search query
        method (str): Search method ("semantic", "keyword", "hybrid")
        k (int): Number of results to return
        threshold (float): Minimum similarity threshold

    Returns:
        list: Ranked search results with scores and metadata
    """

    results = []

    # Semantic Search Implementation
    if method in ["semantic", "hybrid"]:
        # Generate query embedding
        query_vector = embedder.encode([query])
        faiss.normalize_L2(query_vector.astype("float32"))

        # Search in FAISS index
        similarities, indices = doc_system["faiss_index"].search(
            query_vector.astype("float32"), k=k * 2
        )

        # Process semantic results
        for sim, idx in zip(similarities[0], indices[0]):
            if sim > threshold:
                results.append(
                    {
                        "text": doc_system["texts"][idx],
                        "score": float(sim),
                        "method": "semantic",
                        "index": int(idx),
                    }
                )

    # Keyword Search Implementation
    if method in ["keyword", "hybrid"]:
        # Tokenize query for BM25
        query_tokens = re.findall(r"\w+", query.lower())
        bm25_scores = doc_system["bm25_index"].get_scores(query_tokens)

        # Get top results and normalize scores
        top_indices = np.argsort(bm25_scores)[::-1][: k * 2]
        max_score = max(bm25_scores) if max(bm25_scores) > 0 else 1

        # Process keyword results
        for idx in top_indices:
            if bm25_scores[idx] > 0:
                normalized_score = bm25_scores[idx] / max_score
                if normalized_score > threshold:
                    results.append(
                        {
                            "text": doc_system["texts"][idx],
                            "score": float(normalized_score),
                            "method": "keyword",
                            "index": int(idx),
                        }
                    )

    # Remove duplicates and rank results
    seen_indices = set()
    unique_results = []
    for result in sorted(results, key=lambda x: x["score"], reverse=True):
        if result["index"] not in seen_indices:
            unique_results.append(result)
            seen_indices.add(result["index"])

    return unique_results[:k]


questions_search = [
    "rocket propulsion physics",
    "STS-44",  # Answer is 2,890,067 miles
    "Do there exist many worlds ...",  # Complete text phrase: "Do there exist many worlds, or is there but a single world? This is one of the most noble and exalted questions in the study of Nature.""
    "Man hath weaved out a net, and this net throwne ....",  # Complete text phrase: Man hath weaved out a net, and this net throwne upon the Heavens, and now they are his own.""
    "Roger Bacon",  # Complete text: "Roger Bacon, c. 1214 to c. 1292 A monk, Bacon wrote about gunpowder in his The Epistola Fratris R. Baconis, de secretis operibus artis et naturae et nullitate magiae: “We can, with saltpeter and other substances, compose artificially a fire that can be launched over long distances"
    "Project X - 51",  # Section of: "To apply rocket principles and design, construct, test, and launch a water rocket using a real world problem-solving simulation."
]

for question in questions_search:
    # Test the search system
    test_query = question
    search_results = universal_search(test_query, method="hybrid", k=10)
    print("\n\n" + "-" * 75)
    print(f"Search Test - Query: '{test_query}'")
    print(f"Found {len(search_results)} results:")
    for i, result in enumerate(search_results, 1):
        print(f"\n{i}. Score: {result['score']:.3f} | Method: {result['method']}")
        print(f"   Text: {result['text']}")



---------------------------------------------------------------------------
Search Test - Query: 'rocket propulsion physics'
Found 10 results:

1. Score: 1.000 | Method: keyword
   Text: the primary payload on this mission was Spacelab-2. Despite an abort-to-orbit (ato), which required mission replanning, 
the mission was declared a success. a special part of the modular Spacelab system, “the igloo,” located at the head of 
three-pallet train, provided onsite support to instruments mounted on pallets. the main mission objective was to verify 
the performance of Spacelab systems, determine interface capability of the orbiter, measure the environment induced by 
the spacecraft. experiments covered life sciences, plasma physics, astronomy, high-energy astrophysics, solar physics, 
atmospheric physics, and technology research.

2. Score: 0.964 | Method: keyword
   Text: StS-9 carried the first Spacelab mission and the first european Space agency (eSa) astronaut, ulf D. merbold of germany

In [6]:
# =============================================================================
# SECTION 6: RETRIEVAL-AUGMENTED GENERATION (RAG) IMPLEMENTATIONS
# =============================================================================

"""
RAG System Theory and Implementations:

Retrieval-Augmented Generation combines the parametric knowledge of language models
with external knowledge bases to produce more accurate, factual responses.

We implement three sophisticated RAG variants:

1. Simple RAG: Direct retrieval and generation
2. Self-RAG: Self-reflective system with confidence assessment
3. HyDE (Hypothetical Document Embeddings): Uses hypothetical documents for improved retrieval

Each approach addresses different challenges in knowledge-grounded text generation.
"""


# Simple RAG Implementation
def simple_rag(
    query,
    search_method="hybrid",
    max_chars_context=2000,
    max_output_tokens=350,
    k_docs_rag=5,
):
    """
    Simple Retrieval-Augmented Generation implementation.

    This is the foundational RAG approach that:
    1. Retrieves relevant documents using search system
    2. Constructs context from retrieved documents
    3. Generates response using context-augmented prompt

    Args:
        query (str): User query
        search_method (str): Search method to use
        max_chars_context (int): Maximum context characters
        max_output_tokens (int): Maximum tokens in response
        k_docs_rag (int): Number of documents to retrieve

    Returns:
        tuple: (response, number_of_documents_used)
    """

    # Retrieve relevant documents
    results = universal_search(query, method=search_method, k=k_docs_rag)

    if results:
        # Construct context from retrieved documents
        context = "\n\n".join([r["text"] for r in results])[:max_chars_context]
        prompt = f"Based on this information:\n\n{context}\n\nAnswer: {query}"
    else:
        # Fallback to general knowledge
        prompt = f"Answer with general knowledge: {query}"

    # Generate response
    response = generate_response(prompt, max_tokens=max_output_tokens)
    return response, len(results)


# Self-RAG Implementation
def self_rag(query, max_output_tokens=350):
    """
    Self-Reflective Retrieval-Augmented Generation.

    This advanced RAG system includes:
    1. Self-assessment of whether external knowledge is needed
    2. Confidence evaluation of generated responses
    3. Iterative refinement based on confidence scores

    The system uses semantic similarity between query and response
    as a proxy for response quality and relevance.

    Args:
        query (str): User query
        max_output_tokens (int): Maximum tokens in response

    Returns:
        str: Self-validated and potentially refined response
    """

    # Step 1: Assess if external knowledge is needed
    check_prompt = f"Does this question require external documents to answer accurately? Question: {query}\nAnswer: YES or NO"
    needs_docs = generate_response(check_prompt, max_tokens=10)

    if "yes" in needs_docs.lower():
        # Use RAG with retrieved documents
        response, num_docs = simple_rag(query)

        # Step 2: Validate response quality using semantic similarity
        query_emb = embedder.encode([query])
        resp_emb = embedder.encode([response])
        confidence = np.dot(query_emb, resp_emb.T)[0][0] / (
            np.linalg.norm(query_emb) * np.linalg.norm(resp_emb)
        )

        # Step 3: Refine if confidence is low
        if confidence < 0.7:
            refine_prompt = f"Improve this response to better address the question:\nOriginal: {response}\nQuestion: {query}\nImproved:"
            response = generate_response(refine_prompt, max_tokens=max_output_tokens)
    else:
        # Use general knowledge
        response = generate_response(f"Explain: {query}", max_tokens=max_output_tokens)

    return response


# HyDE (Hypothetical Document Embeddings) Implementation
def hyde_rag(
    query,
    max_char_context=1500,
    max_output_tokens=350,
    max_output_hyde_tokens=250,
    k_rag_documents=5,
    rag_min_score=0.5,
):
    """
    Hypothetical Document Embeddings RAG implementation.

    HyDE improves retrieval by:
    1. Generating a hypothetical document that would answer the query
    2. Using this hypothetical document for embedding-based search
    3. Retrieving real documents similar to the hypothetical one
    4. Generating final answer from retrieved real documents

    This approach often finds more relevant documents than direct query embedding,
    as the hypothetical document provides richer semantic context.

    Args:
        query (str): User query
        max_char_context (int): Maximum context characters
        max_output_tokens (int): Maximum tokens in final response
        max_output_hyde_tokens (int): Maximum tokens in hypothetical document
        k_rag_documents (int): Number of documents to retrieve
        rag_min_score (float): Minimum similarity score for retrieval

    Returns:
        str: Response based on HyDE-retrieved documents
    """

    # Step 1: Generate hypothetical document
    hyp_prompt = f"Write a technical paragraph that answers: {query}"
    hypothetical_doc = generate_response(hyp_prompt, max_tokens=max_output_hyde_tokens)

    # Step 2: Use hypothetical document for retrieval
    results = universal_search(
        hypothetical_doc, method="semantic", k=k_rag_documents, threshold=rag_min_score
    )

    if results:
        # Step 3: Generate answer from real retrieved documents
        context = "\n\n".join([r["text"] for r in results])[:max_char_context]
        final_prompt = f"Real information found:\n{context}\n\nOriginal question: {query}\nAnswer based on real information:"
        response = generate_response(final_prompt, max_tokens=max_output_tokens)
    else:
        # Fallback if no relevant documents found
        response = generate_response(f"Answer: {query}", max_tokens=max_output_tokens)

    return response


# Test all RAG implementations
test_queries = [
    "How do rocket engines work in space?",
    "What is the rocket equation?",
    "Explain Newton's laws in rocket propulsion",
]

print("Testing RAG Implementations:")
print("=" * 50)

for i, query in enumerate(test_queries, 1):
    print(f"\nTest {i}: {query}")
    print("-" * 30)

    # Simple RAG
    simple_response, doc_count = simple_rag(query)
    print(f"Simple RAG ({doc_count} docs): {simple_response}")

    # Self-RAG
    self_response = self_rag(query)
    print(f"Self-RAG: {self_response}")

    # HyDE RAG
    hyde_response = hyde_rag(query)
    print(f"HyDE RAG: {hyde_response}")

Testing RAG Implementations:

Test 1: How do rocket engines work in space?
------------------------------
Simple RAG (5 docs): Rocket engines work in space by expelling gases at high speeds through a nozzle, which then accelerates them further. Here's a simplified breakdown of how they function:

1. **Expulsion**: The rocket engine uses a combustion process where fuel and oxidizer mix under high pressure to create a flame. As the flame burns, it creates heat and energy.

2. **Gas Ejection**: The heated gas from the flame travels through a nozzle, creating a high-speed jet of gas. This jet is directed towards the exhaust port of the rocket.

3. **Acceleration**: The high-speed jet of gas continues to accelerate after leaving the nozzle. This acceleration allows the rocket to move away from the Earth's surface.

4. **Thrust**: The thrust generated by the rocket engine pushes the rocket upward, causing it to lift off the ground.

5. **Controlled Flight**: By adjusting the angle of the noz

In [7]:
# =============================================================================
# SECTION 7: DYNAMIC AGENT SYSTEM
# =============================================================================

"""
Dynamic Agent Architecture:

The dynamic agent represents an advanced AI system capable of autonomous
reasoning and action selection. Key components include:

1. Autonomous Decision Making: The agent evaluates situations and selects
   appropriate actions without rigid constraints

2. Memory Management: Maintains context across multiple steps while managing
   information overload

3. Action Execution: Performs various tasks including search, analysis,
   calculation, and content generation

4. Completion Assessment: Determines when sufficient information has been
   gathered to answer the query

This architecture enables sophisticated multi-step reasoning for complex queries
that require information synthesis from multiple sources.
"""


class DynamicAgent:
    """
    Advanced autonomous agent with minimal constraints for complex reasoning.

    This agent can:
    - Make autonomous decisions about next actions
    - Maintain memory across multiple reasoning steps
    - Execute diverse actions (search, analyze, calculate, generate)
    - Self-assess completion status
    - Synthesize information from multiple sources
    """

    def __init__(self, max_steps=10):
        """
        Initialize the dynamic agent.

        Args:
            max_steps (int): Maximum reasoning steps to prevent infinite loops
        """
        self.max_steps = max_steps
        self.memory = []
        self.step_count = 0

    def think_and_act(self, objective):
        """
        Core autonomous reasoning and action loop.

        The agent iteratively:
        1. Analyzes current context and objective
        2. Decides on next action autonomously
        3. Executes the chosen action
        4. Updates memory with results
        5. Assesses if objective is complete

        Args:
            objective (str): High-level objective to accomplish

        Returns:
            dict: Final response with metadata
        """

        print(f"AGENT OBJECTIVE: {objective}")
        print("=" * 50)

        while self.step_count < self.max_steps:
            self.step_count += 1

            # Build context for decision making
            context = self._build_context(objective)

            # Autonomous decision making
            decision = self._make_decision(context)

            print(f"\nStep {self.step_count}")
            print("-" * 20)
            print(f"Thinking: {decision.get('thought', 'Processing')}")

            # Check if agent believes task is complete
            if self._is_complete(decision):
                print("Agent determined task is complete")
                break

            # Execute chosen action
            result = self._execute_action(decision)
            print(f"Action: {result}")

            # Update memory
            self.memory.append(result)

        # Synthesize final response
        return self._synthesize_response(objective)

    def _build_context(
        self, objective, max_previous_turns=10, max_total_chars_context=1500
    ):
        """
        Build dynamic context for decision making.

        Args:
            objective (str): Original objective
            max_previous_turns (int): Maximum previous actions to include
            max_total_chars_context (int): Maximum total context characters

        Returns:
            dict: Context dictionary for decision making
        """
        recent_memory = self.memory[-max_previous_turns:] if self.memory else []
        return {
            "objective": objective,
            "memory": recent_memory,
            "step": self.step_count,
            "total_info": (
                " ".join(self.memory)[-max_total_chars_context:] if self.memory else ""
            ),
        }

    def _make_decision(self, context, max_tokens_response=250):
        """
        Autonomous decision making using language model reasoning.

        The agent evaluates the current context and autonomously decides
        what action to take next. This represents true autonomous reasoning
        without rigid decision trees.

        Args:
            context (dict): Current context and memory
            max_tokens_response (int): Maximum tokens for decision

        Returns:
            dict: Decision with thought process and action plan
        """

        decision_prompt = f"""You are an autonomous agent. Analyze and decide your next action.

        OBJECTIVE: {context['objective']}
        CURRENT STEP: {context['step']}
        PREVIOUS ACTIONS: {context['memory'] if context['memory'] else 'None'}

        Available capabilities:
        - Search documents for information
        - Analyze existing information  
        - Generate new insights
        - Make calculations
        - Any other logical action

        Decide what to do next. If you have sufficient information to fully answer the objective, say COMPLETE.

        Think freely and decide:"""

        decision_text = generate_response(
            decision_prompt, max_tokens=max_tokens_response
        )

        # Parse decision flexibly
        thought = decision_text if decision_text else "Thinking..."
        action_type = self._extract_action_type(decision_text)
        parameters = self._extract_parameters(decision_text, context["objective"])

        return {
            "thought": thought,
            "action": action_type,
            "params": parameters,
            "raw": decision_text,
        }

    def _extract_action_type(self, text):
        """
        Extract action type from agent's decision text.

        Args:
            text (str): Agent's decision text

        Returns:
            str: Identified action type
        """
        text_lower = text.lower()

        if any(word in text_lower for word in ["search", "find", "look", "retrieve"]):
            return "search"
        elif any(word in text_lower for word in ["analyze", "examine", "evaluate"]):
            return "analyze"
        elif any(word in text_lower for word in ["calculate", "compute", "math"]):
            return "calculate"
        elif any(word in text_lower for word in ["generate", "create", "produce"]):
            return "generate"
        else:
            return "search"  # Default safe action

    def _extract_parameters(self, text, objective):
        """
        Extract action parameters from decision text.

        Args:
            text (str): Agent's decision text
            objective (str): Original objective as fallback

        Returns:
            str: Extracted parameters for action
        """
        lines = text.split("\n")
        for line in lines:
            if ":" in line and len(line.strip()) > 10:
                return line.split(":", 1)[1].strip()
        return objective  # Fallback to objective

    def _execute_action(self, decision, k_rag_docs=8, rag_max_chars=1000):
        """
        Execute the agent's chosen action.

        Args:
            decision (dict): Agent's decision with action and parameters
            k_rag_docs (int): Number of documents for RAG
            rag_max_chars (int): Maximum characters for analysis

        Returns:
            str: Result of action execution
        """

        action = decision["action"]
        params = decision["params"]

        if action == "search":
            # Execute search action
            results = universal_search(params, method="hybrid", k=k_rag_docs)
            if results:
                return f"Found information: {results[0]['text']}"
            else:
                return "No relevant information found"

        elif action == "analyze":
            # Execute analysis action
            analysis_prompt = f"Analyze this information: {params[:rag_max_chars]}"
            return generate_response(analysis_prompt, max_tokens=100)

        elif action == "calculate":
            # Execute calculation action
            numbers = re.findall(r"\d+\.?\d*", params)
            if len(numbers) >= 2:
                try:
                    result = float(numbers[0]) * float(numbers[1])
                    return f"Calculation result: {result}"
                except:
                    return "Calculation completed"
            return "Mathematical analysis completed"

        elif action == "generate":
            # Execute generation action
            gen_prompt = f"Generate information about: {params}"
            return generate_response(gen_prompt, max_tokens=120)

        else:
            # Generic processing
            process_prompt = f"Process this request: {params}"
            return generate_response(process_prompt, max_tokens=100)

    def _is_complete(self, decision):
        """
        Check if agent believes task is complete.

        Args:
            decision (dict): Agent's current decision

        Returns:
            bool: True if task is complete
        """
        return any(
            word in decision["raw"].lower()
            for word in ["complete", "finished", "done", "sufficient"]
        )

    def _synthesize_response(self, objective, max_chars_context=2000):
        """
        Synthesize final comprehensive response from all gathered information.

        Args:
            objective (str): Original objective
            max_chars_context (int): Maximum context characters

        Returns:
            dict: Final response with metadata
        """

        all_info = " ".join(self.memory)

        synthesis_prompt = f"""Based on all the work done, provide a comprehensive answer:

        ORIGINAL OBJECTIVE: {objective}
        INFORMATION GATHERED: {all_info[:max_chars_context]}
        STEPS COMPLETED: {self.step_count}

        Provide a complete, well-structured final answer:"""

        final_response = generate_response(synthesis_prompt, max_tokens=500)

        print("\n" + "=" * 50)
        print("FINAL AGENT RESPONSE:")
        print("=" * 50)
        print(final_response)

        return {
            "response": final_response,
            "steps": self.step_count,
            "info_gathered": len(all_info),
        }


# Simplified interface for dynamic agent
def dynamic_agent(objective):
    """
    Simplified interface for the dynamic agent system.

    Args:
        objective (str): Complex objective requiring multi-step reasoning

    Returns:
        dict: Agent's final response with metadata
    """
    agent = DynamicAgent(max_steps=6)
    return agent.think_and_act(objective)


################
# Test the dynamic agent

question_to_agent = [
    "Explain the physics principles behind rocket propulsion and how they enable space travel",
    "Explain the mision STS-55 without inventing information",
    "Explain the mision STS-128 without inventing information",
    "Explain the 'Mars Research, Testbeds and Missions' information",
    "Explain the NASA guiding principles of exploration",
]

for question in question_to_agent:
    test_objective = question
    print("\n\n" + "-" * 75)
    agent_result = dynamic_agent(test_objective)



---------------------------------------------------------------------------
AGENT OBJECTIVE: Explain the physics principles behind rocket propulsion and how they enable space travel

Step 1
--------------------
Thinking: I'm ready to assist with this objective. Please provide me with more details or context so I can start analyzing it.
Action: Found information: spacecraft are on their way into interstellar space 
as you read this. Someday, they will be followed by 
human explorers. 
Often lost in the shadows of time, early rocket 
pioneers “pushed the envelope” by creating rocket-
propelled devices for land, sea, air, and space. 
When the scientific principles governing motion 
were discovered, rockets graduated from toys and 
novelties to serious devices for commerce, war, 
travel, and research. This work led to many of the 
most amazing discoveries of our time. 
The vignettes that follow provide a small sampling 
of stories from the history of rockets. They form a 
rocket time lin

In [8]:
# =============================================================================
# SECTION 8: SPECIALIZED APPLICATION SYSTEMS
# =============================================================================

"""
Specialized Application Theory:

Building on the foundational RAG and agent systems, we create specialized
applications that automatically select the most appropriate AI technique
based on query characteristics. This meta-learning approach optimizes
performance by matching methods to use cases.

Query Classification Logic:
- Diagnostic queries → Self-RAG (for self-validation)
- Comparative queries → Dynamic Agent (for multi-perspective analysis)
- General queries → Simple RAG (for efficiency)

This approach demonstrates how AI systems can become more sophisticated
by learning to choose appropriate sub-systems for different tasks.
"""


def intelligent_assistant(query):
    """
    Intelligent assistant that automatically selects optimal AI method.

    This system demonstrates meta-learning by analyzing query characteristics
    and selecting the most appropriate underlying AI technique:

    - Problems/diagnostics: Uses Self-RAG for validation
    - Comparisons/analysis: Uses Dynamic Agent for comprehensive reasoning
    - General questions: Uses efficient Simple RAG

    Args:
        query (str): User query requiring intelligent processing

    Returns:
        str: Response from most appropriate AI system
    """

    query_lower = query.lower()

    # Route to Self-RAG for diagnostic queries
    if any(
        word in query_lower
        for word in ["problem", "issue", "noise", "fault", "troubleshoot"]
    ):
        print("Selected method: Self-RAG (diagnostic query)")
        return self_rag(query)

    # Route to Dynamic Agent for comparative analysis
    elif any(
        word in query_lower
        for word in ["compare", "difference", "versus", "vs", "analysis"]
    ):
        print("Selected method: Dynamic Agent (comparative query)")
        result = dynamic_agent(query)
        return result["response"] if isinstance(result, dict) else result

    # Route to Simple RAG for general queries
    else:
        print("Selected method: Simple RAG (general query)")
        response, _ = simple_rag(query)
        return response


def research_agent(topic):
    """
    Comprehensive research agent for in-depth topic analysis.

    This specialized agent conducts multi-perspective research by:
    1. Breaking down complex topics into components
    2. Gathering information from multiple angles
    3. Synthesizing findings into comprehensive reports

    Args:
        topic (str): Research topic requiring comprehensive analysis

    Returns:
        dict: Comprehensive research results
    """

    research_objective = f"Conduct comprehensive research on: {topic}. Include multiple perspectives, technical details, and practical implications."
    return dynamic_agent(research_objective)


# Test specialized systems
test_queries = [
    "My spacecraft is experiencing thrust problems during launch",
    "Compare chemical rockets versus ion propulsion systems",
    "How does orbital mechanics work?",
]

print("Testing Intelligent Assistant:")
print("=" * 40)

for i, query in enumerate(test_queries, 1):
    print("\n\n" + f"Test {i}: {query}")
    print("-" * 30)
    response = intelligent_assistant(query)
    print(f"Response: {response}")


################
# Test research agent
print("\n" + "=" * 40)
print("Testing Research Agent:")
print("=" * 40)

question_to_agent = [
    "Explain the physics principles behind rocket propulsion and how they enable space travel",
    "Explain the mision STS-55 without inventing information",
    "Explain the mision STS-128 without inventing information",
    "Explain the 'Mars Research, Testbeds and Missions' information",
    "Explain the NASA guiding principles of exploration",
]

for question in question_to_agent:
    research_topic = question
    print("\n\n" + "-" * 75)
    research_result = research_agent(research_topic)

Testing Intelligent Assistant:


Test 1: My spacecraft is experiencing thrust problems during launch
------------------------------
Selected method: Self-RAG (diagnostic query)
Response: I'm sorry to hear that your spacecraft is having issues with thrust during its launch. Thrust refers to the force generated by engines or other propulsion systems designed to move an object through space. If there's a problem with the thrust system, it could be due to several factors:

1. Engine failure: The most common cause of thrust problems is a malfunctioning engine. This can happen for various reasons such as mechanical failures, electrical issues, or software errors.

2. Fuel supply issues: If the fuel supply to the engines is interrupted or insufficient, the engines may not be able to generate enough thrust to overcome gravity and achieve the desired trajectory.

3. Ignition sequence: Incorrect ignition order or timing can also lead to poor thrust performance.

4. Environmental conditions: Extr

In [9]:
# =============================================================================
# SECTION 9: SYSTEM VALIDATION AND PERFORMANCE ANALYSIS
# =============================================================================

"""
Validation Methodology:

Comprehensive validation ensures our AI systems perform reliably across
diverse scenarios. We test multiple dimensions:

1. Factual Accuracy: Ability to provide correct information
2. Relevance: Responses address the actual query
3. Coherence: Logical flow and readability
4. Completeness: Comprehensive coverage of topics
5. Method Selection: Appropriate technique selection

This validation approach ensures practical utility while identifying
areas for improvement in real-world applications.
"""


def comprehensive_system_validation():
    """
    Comprehensive validation of all AI systems with diverse test cases.

    Tests multiple categories:
    - Scientific/Technical queries
    - Creative/Analytical tasks
    - Problem-solving scenarios
    - Comparative analysis requests

    Returns:
        dict: Validation results and performance metrics
    """

    validation_test_cases = [
        {
            "query": "Explain the rocket equation and its significance in space exploration",
            "category": "Scientific/Technical",
            "expected_method": "Simple RAG",
        },
        {
            "query": "Compare the advantages and disadvantages of chemical vs electric propulsion",
            "category": "Comparative Analysis",
            "expected_method": "Dynamic Agent",
        },
        {
            "query": "My rocket engine is showing irregular thrust patterns",
            "category": "Problem Solving",
            "expected_method": "Self-RAG",
        },
        {
            "query": "What are the challenges of interplanetary travel?",
            "category": "General Knowledge",
            "expected_method": "Simple RAG",
        },
        {
            "query": "Analyze the future prospects of Mars colonization technology",
            "category": "Research/Analysis",
            "expected_method": "Dynamic Agent",
        },
    ]

    print("COMPREHENSIVE SYSTEM VALIDATION")
    print("=" * 50)

    results = []

    for i, test_case in enumerate(validation_test_cases, 1):
        print(f"\nTest Case {i}: {test_case['category']}")
        print(f"Query: {test_case['query']}")
        print(f"Expected Method: {test_case['expected_method']}")
        print("-" * 40)

        # Test intelligent assistant
        response = intelligent_assistant(test_case["query"])

        # Evaluate response length and coherence
        response_length = len(response.split())
        has_technical_content = any(
            term in response.lower()
            for term in ["equation", "physics", "energy", "force", "velocity", "mass"]
        )

        result = {
            "test_case": i,
            "category": test_case["category"],
            "response_length": response_length,
            "has_technical_content": has_technical_content,
            "response_preview": (
                response[:200] + "..." if len(response) > 200 else response
            ),
        }

        results.append(result)

        print(f"Response Length: {response_length} words")
        print(f"Technical Content: {has_technical_content}")
        print(f"Response Preview: {result['response_preview']}")

    # Calculate overall metrics
    avg_response_length = np.mean([r["response_length"] for r in results])
    technical_content_rate = np.mean([r["has_technical_content"] for r in results])

    print("\n" + "=" * 50)
    print("VALIDATION SUMMARY")
    print("=" * 50)
    print(f"Total Test Cases: {len(results)}")
    print(f"Average Response Length: {avg_response_length:.1f} words")
    print(f"Technical Content Rate: {technical_content_rate:.1%}")
    print(
        f"System Successfully Processed: {len(results)}/{len(validation_test_cases)} queries"
    )

    return {
        "results": results,
        "metrics": {
            "avg_response_length": avg_response_length,
            "technical_content_rate": technical_content_rate,
            "success_rate": len(results) / len(validation_test_cases),
        },
    }


# Run comprehensive validation
validation_results = comprehensive_system_validation()

COMPREHENSIVE SYSTEM VALIDATION

Test Case 1: Scientific/Technical
Query: Explain the rocket equation and its significance in space exploration
Expected Method: Simple RAG
----------------------------------------
Selected method: Simple RAG (general query)
Response Length: 287 words
Technical Content: True
Response Preview: The rocket equation is a fundamental principle in rocket science that describes how a rocket's trajectory changes as it burns fuel. The equation states that the velocity (v) of a rocket increases with...

Test Case 2: Comparative Analysis
Query: Compare the advantages and disadvantages of chemical vs electric propulsion
Expected Method: Dynamic Agent
----------------------------------------
Selected method: Dynamic Agent (comparative query)
AGENT OBJECTIVE: Compare the advantages and disadvantages of chemical vs electric propulsion

Step 1
--------------------
Thinking: COMPLETE. I don't have access to current scientific data or specific comparisons between chemical

____

References

In [10]:
print(
    """
ACADEMIC REFERENCES AND CITATIONS

This educational implementation builds upon foundational research in 
information retrieval, natural language processing, and AI systems:

1. Retrieval-Augmented Generation:
   Lewis, P., et al. (2020). "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks." 
   arXiv:2005.11401. https://arxiv.org/abs/2005.11401

2. Dense Passage Retrieval:
   Karpukhin, V., et al. (2020). "Dense Passage Retrieval for Open-Domain Question Answering."
   arXiv:2004.04906. https://arxiv.org/abs/2004.04906

3. Self-RAG:
   Asai, A., et al. (2023). "Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection."
   arXiv:2310.11511. https://arxiv.org/abs/2310.11511

4. HyDE (Hypothetical Document Embeddings):
   Gao, L., et al. (2022). "Precise Zero-Shot Dense Retrieval without Relevance Labels."
   arXiv:2212.10496. https://arxiv.org/abs/2212.10496

5. BM25 Algorithm:
   Robertson, S., & Zaragoza, H. (2009). "The Probabilistic Relevance Framework: BM25 and Beyond."
   Foundations and Trends in Information Retrieval, 3(4), 333-389.

6. Sentence Transformers:
   Reimers, N., & Gurevych, I. (2019). "Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks."
   arXiv:1908.10084. https://arxiv.org/abs/1908.10084

7. FAISS (Facebook AI Similarity Search):
   Johnson, J., Douze, M., & Jégou, H. (2019). "Billion-scale similarity search with GPUs."
   IEEE Transactions on Big Data, 7(3), 535-547.

TECHNICAL LIBRARIES AND FRAMEWORKS:

- HuggingFace Transformers: https://huggingface.co/docs/transformers/
- Sentence Transformers: https://www.sbert.net/
- LangChain: https://python.langchain.com/
- FAISS: https://faiss.ai/
- PyTorch: https://pytorch.org/
- NumPy: https://numpy.org/

DATA SOURCES:

This educational implementation uses publicly available NASA educational materials
as demonstration data, in accordance with NASA's open data policies and
educational mission. All NASA content is used for educational purposes only.

NASA Open Data Policy: https://www.nasa.gov/open/
NASA Educational Resources: https://www.nasa.gov/audience/foreducators/

DISCLAIMER:

This implementation is developed solely for educational and research purposes.
It is not intended for commercial use, production deployment, or any application
requiring guaranteed performance or reliability. The code serves as a learning
resource for understanding modern AI techniques in information retrieval and
text generation.

Users should refer to original research papers and official documentation
for production implementations and cite appropriate sources when using
these techniques in academic or commercial contexts.
"""
)


ACADEMIC REFERENCES AND CITATIONS

This educational implementation builds upon foundational research in 
information retrieval, natural language processing, and AI systems:

1. Retrieval-Augmented Generation:
   Lewis, P., et al. (2020). "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks." 
   arXiv:2005.11401. https://arxiv.org/abs/2005.11401

2. Dense Passage Retrieval:
   Karpukhin, V., et al. (2020). "Dense Passage Retrieval for Open-Domain Question Answering."
   arXiv:2004.04906. https://arxiv.org/abs/2004.04906

3. Self-RAG:
   Asai, A., et al. (2023). "Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection."
   arXiv:2310.11511. https://arxiv.org/abs/2310.11511

4. HyDE (Hypothetical Document Embeddings):
   Gao, L., et al. (2022). "Precise Zero-Shot Dense Retrieval without Relevance Labels."
   arXiv:2212.10496. https://arxiv.org/abs/2212.10496

5. BM25 Algorithm:
   Robertson, S., & Zaragoza, H. (2009). "The Probabilistic Relevance Fram